<a href="https://colab.research.google.com/github/jshogland/SpatialModelingTutorials/blob/main/Notebooks/NAFEW_sample_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NAFEW: Sample Design
## In this notebook we will discuss sample designs and create a well spread and balanced sample to estimate live basal area for the Missoula District of the Lolo national forest using [Raster Tools](https://github.com/UM-RMRS/raster_tools) and remotely sensed imagery
### Objectives
- Learn about sample designs
- Learn how to access remotely sensed data
- Learn how to create a well spread and balanced sample

### Datasets
- Forest Boundary (OSM)
- Sentinel 2 imagery (STAC, plantary computer)
- USGS elevation datasets (USGS)
- Basal Area $ft^2$ [Google Earth Engine TreeMap](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_TreeMap_v2022)

### Sections
1. Installing software
2. Importing packages
3. Downloading Data
4. Convert pixels into points
5. Sampling (random, systematic, stratified, & spread and balance)
6. Space filling curves
7. Evaluate sample designs
8. Create a well spread and balanced sample
9. Accessibility

by John Hogland 6/15/2026

## Install software

In [ ]:
!pip install mapclassify
!pip install osmnx
!pip install raster_tools
!pip install planetary-computer
!pip install pystac-client
!pip install stackstac
!pip install hilbertcurve
!pip install py3dep

## Import packages

In [ ]:
#get packages
import osmnx as ox, planetary_computer, pystac_client, stackstac
import geopandas as gpd, pandas as pd, os, numpy as np, py3dep

from hilbertcurve.hilbertcurve import HilbertCurve
from raster_tools import Raster,zonal, surface
import raster_tools as rt

import ee, geemap

## Authenticate EE

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='ee-jshogland') #you will want to select your personal cloud project

## Look at the response variable from Google Earth Engine (Treemap Basal Area)
Select the Google Earth Engine resource TreeMap\BALIVE. This will be what we want to estimate.

In [ ]:
resource="USFS/GTAC/TreeMap/v2022"
dsname='BALIVE'
dataset = ee.ImageCollection(resource)
treemap=dataset.select(dsname)
img=treemap.mosaic()

Show the map

In [ ]:
treemap_vis = {
    'min': 1.0,
    'max': 250,
    "palette": 'Greens'
}

m = geemap.Map()
m.set_center(-113.82, 46.72, 6)
m.add_layer(treemap, treemap_vis, 'Basal Area')
m

## Download Data
- Get the NF district boundary
- Get Sentinel 2 imagery
- Get Elevation data
- Get Basal from GEE ($ft^2/acre$)

### Create Definitions to download stack data

In [ ]:
#create definition to mosaic stac data
def mosaic_stac(xr):
    return stackstac.mosaic(xr)

#create definition to extract stac data
def get_stac_data(geo,url="https://planetarycomputer.microsoft.com/api/stac/v1",name="sentinel-2-l2a",res=30,crs=5070,**kwarg):
    '''
    gets tiled data from planetary computer as a dask backed xarray that intersects the geometry of the point, line, or polygon

    geo = (polygon) geometry bounding box (WGS84)
    url = (string) base url to planetary computer https://planetarycomputer.microsoft.com/api/stac/v1
    name = (string) catelog resource
    res = (tuple of numbers) output resolution (x,y)
    crs = (int) output crs
    kwarg = additional keyword arguments to pass to the stac search, e.g.,
        datetime = (string) data time intervale e.g., one month: 2023-06, range: 2023-06-02/2023-06-17
        limit = (int) max number of items to return

    returns (xarray data array and stac item catalog)
    '''
    catalog = pystac_client.Client.open(url, modifier=planetary_computer.sign_inplace)
    srch = catalog.search(collections=name, intersects=geo, **kwarg)
    ic = srch.item_collection()
    if(len(ic.items)>0):
        xra = stackstac.stack(ic,resolution=res,epsg=crs)
        xra = mosaic_stac(xra)
    else:
        xra=None

    return xra,ic

### Download national forest boundary
We will use OSM to download National Forest district boundary and further select one of the geometries to serve as our study area


In [ ]:
nf=ox.geocode_to_gdf('Lolo National Forest, MT, USA',)
nfe=nf.explode(ignore_index=True)
bnd=nfe.iloc[[26]] #get the 26 geometry for the boundary
bnd.explore() #look at the study area boundary

#How many acres is the study area?
#print('Acres in study area:',(bnd.to_crs(5070).area/4046.86).iloc[0])

### Download Sentinel 2 (2021-07), elevation, and basal area
2022 is the most recent TreeMap Basal area map so let's get a Sentinel 2 image from July 2021.

In [ ]:
#project and buffer boundary by 200 meters to ensure we get all data that intersects the boundary
bndp=bnd.to_crs(5070)
bndpb=bndp.buffer(200)
bndb=bndpb.to_crs(4326)

#get stac data landsat data
out_sent_path='sent2_2021.tif'
if(not os.path.exists(out_sent_path)):
    xmin,ymin,xmax,ymax=bndp.total_bounds
    sent, ic =get_stac_data(bndb.geometry.iloc[0],"https://planetarycomputer.microsoft.com/api/stac/v1",name="sentinel-2-l2a",res=10,crs=5070,datetime='2021-07-21/2021-07-23',limit=1000)
    sents=Raster(sent.sel(band=['B02', 'B03', 'B04','B08', 'B11','B12', 'B05'],x=slice(xmin,xmax),y=slice(ymax,ymin)))
    # need to clip raster to study area boundary
    sents.save(out_sent_path, tiled=True)

#Load rasters from disk
sents=Raster(out_sent_path)

#get elevation data
out_elev_path='elev.tif'
if(not os.path.exists(out_elev_path)): #if the elevation file does not exits,
    elev=Raster(py3dep.get_dem(bndb.geometry.iloc[0],resolution=10,crs=4326)).reproject(sents.geobox)
    elev.save(out_elev_path, tiled=True)

#Load rasters from disk and make surface rasters
elev=Raster(out_elev_path)
slope=surface.slope(elev)
north=surface.northing(elev)
east=surface.easting(elev)
curv=surface.curvature(elev)

#get basal area from GeeMap
baa_rs_path='baa.tif'
if(not os.path.exists(baa_rs_path)):

    roi=ee.Geometry.Rectangle(list(bnd.total_bounds))
    geemap.ee_export_image(img,baa_rs_path,region=roi,scale=30)

#Load rasters from disk
baa_rs=Raster(baa_rs_path).reproject(sents.geobox)



### Look at downloaded data

In [ ]:
#stack the rasters
pop_rs=rt.stack_bands([sents,elev,slope,north,east,curv,baa_rs])
pop_rs=rt.clipping.clip(bndp,pop_rs)

#plot the rasters bands
#pop_rs.plot(x='x',y='y',col='band',col_wrap=3,figsize=(15,pop_rs.nbands),robust=True,cmap='PRGn')

#which bands are which?
band_names={1:'B02', 2:'B03', 3:'B04', 4:'B08', 5:'B11', 6:'B12', 7:'B05', 8:'elevation',9:'slope',10:'northing',11:'easting',12:'curvature', 13:'baa'}
print(band_names)

#reset the individual rasters to the clipped rasters
baa_rs=pop_rs.get_bands([13])
sents=pop_rs.get_bands([1,2,3,4,5,6,7])
elev=pop_rs.get_bands([8])
slope=pop_rs.get_bands([9])
north=pop_rs.get_bands([10])
east=pop_rs.get_bands([11])
curv=pop_rs.get_bands([12])

#how many pixels are in the study area?
N=(pop_rs.mask[0]==0).sum().compute()
print('Pixels in study area:',N)

#how many acres are in the study area?
acres=N*100*0.00024711
print('Acres in study area from pixels:',acres)
print('Acres in study area from boundary:',(bndp.area*0.00024711).iloc[0]) #Why did we use bndp instead of bndpb?

#What is our population?
#What is our observational unit?

### Summary statistics for Basal Area.
These are our population statistics (the values we will try to estimate from our sample).

In [ ]:
mean_baa=baa_rs.mean().compute()
std_baa=baa_rs.std().compute()
bands,rows,columns=baa_rs.shape
print('Mean BAA:',mean_baa)
print('Std BAA:',std_baa)
print('Min BAA:',baa_rs.min().compute())
print('Max BAA:',baa_rs.max().compute())
print('Numbers of pixels in the BAA image:',rows*columns)
print('Numbers of pixels in the study area:',N)
print('Null Data value:',baa_rs.null_value)

#are min, max, mean, and std of canopy cover estimates for the entire image or just the study area?
#Does the sentinel 2 data have a mask?
#What if we determined the mean, std, min, and max for the sentinel 2 data? Would that be for the entire image or just the study area?
#What about the elevation data?

In [ ]:
#to make a histogram from our Canopy Cover surface we need to turn the null value to a numpy nan number.
baa_rs.xdata.where(~baa_rs.to_null_mask(),np.nan).plot.hist(figsize=(15,5),bins=20,alpha=0.7) #histogram of Canopy Cover

### Summary statistics for ancillary data
- These are the values we can potentially use to reduce the variation associated with the sampling scheme.

In [ ]:
#get stats for the entire image

st_lst=[]
for b in range(pop_rs.nbands-1): #number of bands minus one because the last band is our response variable
    trs=pop_rs.get_bands([b+1])
    st_lst.append([trs.mean().compute(), trs.std().compute(), trs.min().compute(), trs.max().compute()])

st_df=pd.DataFrame(st_lst,columns=['mean','std','min','max'],index=[band_names[i+1] for i in range(pop_rs.nbands-1)])
print('Statistics:')
display(st_df)

# Why is it important to mask out (clip) area that are not in the study area when calculating statistics?

##

### Correlation among surfaces

In [ ]:
tsmp=bndp.sample_points(1000).explode() #simple random sample of 1000 location within the study area
tvls=zonal.extract_points_eager(tsmp,pop_rs,column_name='band',axis=True).compute()
tvls.columns=band_names.values()
tvls['x']=tsmp.geometry.x.values
tvls['y']=tsmp.geometry.y.values
tvls.corr()

#Which bands are correlated with baa (abs(r)>0.5?
#What would a scatter plot matrix of these variables look like?

## Simple random vs systematic ordered sampling
If we can order our data in such a way that naturally orders our response variable and systematically sample from the population, we can significantly reduce the variability among samples and improve our ability to estimate the population.  
- Simple random sample possible number of samples without replacement $ N!/n!(N-n)!$
- Systematic random sample (ordered) possible number of samples without replacement $ N/n $
- What about variance?

##### Create a sorted list of all BAA cells

In [ ]:
vls_baa=baa_rs.to_numpy().ravel()
vls_baa=vls_baa[vls_baa>=0]
vls_baa.sort()


##### Estimate mean BAA for 1000 iterations from sample sizes of 150 for a simple random sample (SRS) and systematic ordered random sample (SYSO).

In [ ]:
m_lst=[]
sk=vls_baa.shape[0]/150
for i in range(1000):
    m_lst.append([np.random.choice(vls_baa,150).mean(),'srs'])
    rstart=(np.random.random(1)*sk)
    vind=np.arange(rstart,vls_baa.shape,sk).astype('uint64')
    m_lst.append([vls_baa[vind].mean(),'syso'])

tdf=pd.DataFrame(m_lst,columns=['mean_baa','stype'])

##### Look at the distribution of mean estimates
- mean of mean estimates is the mean of the population
- standard deviation of mean estimates is the standard error of the mean

In [ ]:
grps=tdf.groupby('stype')
print('Population mean:',mean_baa)
display(grps.mean_baa.agg(['mean','std','count']))

##### Visualize box plots

In [ ]:
p1=tdf.plot(kind='box',by='stype',figsize=(15,5))
p1.iloc[0].scatter(1, mean_baa, color='red', marker='o', s=100, label=f"Mean: {mean_baa:.2f}")
p1.iloc[0].scatter(2, mean_baa, color='red', marker='o', s=100, label=f"Mean: {mean_baa:.2f}")
p1.iloc[0].set_title('SRS - SYSO')
display(p1)

##### Visualize Histogram

In [ ]:
p2=tdf.groupby('stype')['mean_baa'].plot(kind='hist', legend=True,alpha=0.4,figsize=(15,5))
display(p2)

##### KS statistics for SRS and SYSO and histograms

In [ ]:
import scipy
import scipy.stats

tdf=pd.DataFrame(vls_baa,columns=['BAA'])
tdf['stype']='POP'

bsrs=tdf.sample(10000)

srs_tdf=tdf.sample(150)
srs_tdf['stype']='SRS'

sys_tdf=tdf.iloc[vind]
sys_tdf['stype']='SYSO'

adf=pd.concat([bsrs,srs_tdf,sys_tdf])
p2=adf.groupby('stype')['BAA'].plot(kind='kde', legend=True,linewidth=1,figsize=(15,5),title='Sample Spreads')
display(p2)

#look at KS statistic for distance values bsmp_s vs smp_t
dks_test=scipy.stats.ks_2samp(vls_baa,srs_tdf.BAA) #KS 2 sample test
print('KS test stat for SRS =',dks_test.statistic,'\nKS pvalue for SRS =',dks_test.pvalue)

dks_test=scipy.stats.ks_2samp(vls_baa,sys_tdf.BAA) #KS 2 sample test
print('\nKS test stat for SYSO =',dks_test.statistic,'\nKS pvalue for SYSO =',dks_test.pvalue)

## Evaluating Sampling strategies using ancillary data
Let's assume we are interested in estimating mean BAA for our study site. Additionally, let's also assume we only have time & budget to visit 150 locations within our study site to estimate that mean BAA. To evaluate 4 sampling strategies we will use mean estimates of BAA for 1000 iterations of each sampling strategy. For each iteration, 150 BAA values will be extracted from the BAA surface and summarized to estimate the mean BAA of the study site. Sampling strategies evaluate include:
- simple random sampling
- systematic random sample in space
- stratified random sampling using ancillary data (sentinel reflectance and elevation derivatives)
- Spread and balanced sampling with space filling curves

### Simple random sampling - random in space


#### One example of the spatial layout of a simple random sample for our study site

In [ ]:
n=150
iters=1000
srs=bndp.sample_points(n)
srs.explore()

#### 1000 mean estimates of BAA derived from a simple random sample

In [ ]:
srs_arr=np.empty((iters,2),dtype='object')
for i in range(iters):
    tsrs=bndp.sample_points(n).explode()
    vls=zonal.extract_points_eager(tsrs,baa_rs,'value',axis=1).compute()
    srs_arr[i,0]=vls.mean().iloc[0]
    srs_arr[i,1]='SRS'

### Systematic random sampling - spreading samples in geographic space

#### One example of spatial layout

In [ ]:
tarea=bndp.area.values
carea=tarea/n
sk=carea**0.5
xmin,ymin,xmax,ymax=bndp.total_bounds
rsx,rsy=np.random.random(2)*sk[0]
xs=np.arange(xmin+rsx,xmax,sk[0])
ys=np.arange(ymin+rsy,ymax,sk[0])
xs2,ys2=np.meshgrid(xs,ys)
geom=gpd.GeoSeries.from_xy(xs2.ravel(),ys2.ravel(),crs=bndp.crs)
geoc=geom.clip(bndp)
geoc.explore()

#### 1000 mean estimates of BAA derived from systematic random sampling

In [ ]:
tarea=bndp.area.values
carea=tarea/n
sk=carea**0.5
xmin,ymin,xmax,ymax=bndp.total_bounds

sys_arr=np.empty((iters,2),dtype='object')
for i in range(iters):
    rsx,rsy=np.random.random(2)*sk[0]
    xs=np.arange(xmin+rsx,xmax,sk[0])
    ys=np.arange(ymin+rsy,ymax,sk[0])
    xs2,ys2=np.meshgrid(xs,ys)
    geom=gpd.GeoSeries.from_xy(xs2.ravel(),ys2.ravel(),crs=bndp.crs)
    geoc=geom.clip(bndp)
    vls=zonal.extract_points_eager(geoc,baa_rs,'value',axis=1).compute()
    sys_arr[i,0]=vls.mean().iloc[0]
    sys_arr[i,1]='SYS'


### Stratified Random Sampling - grouping based on ancillary data (Sentinel 2 and Elevation derivatives) and randomly sampling within groups.
The idea behind stratified random sampling is that we can group like areas and sample within those groups to minimize sampling variability. To make groups of similar values we will be using K-means clustering of the sentinel 2 imagery and elevation derivatives. The assumption in this case is that groups derived from the k-mean process are related to BAA and that the variability within groups is less than the total variability in BAA across the study area.
- How to make groups?
- How many observations in each group?
- How to estimate population means?

#### K-mean clustering  
We will be using [Scikit-learn's K-means](https://scikit-learn.org/stable/modules/clustering.html#k-means) classes and functions to create stratum. To build our modeling dataset let's build a training dataset based on systematic random sampling of approximately 100,000 observations so that we ensure our sample is well spread in geographic space. When populating that training set with ancillary data value we really only want data that is somehow related to basal area estimates. Originally we thought that all Sentinel 2 and elevation derivatives provide insight into basal area per acre (BAA). However, our person's correlation statistics suggest that only Sentinel 2 bands B04, B11, and B12 are somewhat correlated with BAA. To create our clusters let's subset our potential predictors down to just B04, B11, and B12.

##### Create the training sample

In [ ]:
#systematic sample of 100,000 locations
tarea=bndp.area.values
carea=tarea/100000
sk=carea**0.5
xmin,ymin,xmax,ymax=bndp.total_bounds

rsx,rsy=np.random.random(2)*sk[0]
xs=np.arange(xmin+rsx,xmax,sk[0])
ys=np.arange(ymin+rsy,ymax,sk[0])
xs2,ys2=np.meshgrid(xs,ys)
geom=gpd.GeoSeries.from_xy(xs2.ravel(),ys2.ravel(),crs=bndp.crs)
geoc=geom.clip(bndp).reset_index(drop=True)

#subset the pop_rs to just the bands we want
sub_rs=pop_rs.get_bands([3,5,6,13])

#extract values for each location
vls=zonal.extract_points_eager(geoc,sub_rs,'value',axis=1).compute()
vls.columns=['B04','B11','B12','baa']

##### Creating the K-means model
- normalize the data
- build the K-mean model
- apply the model to the data
- visualize the results

In [ ]:
#load sklearn libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline

In [ ]:
#subset the data to the predictor variables
X=vls[['B04','B11','B12']].dropna()

#make a pipeline that combines MinMaxScaler normalization with K-means
pline=Pipeline([
    ('scaler',MinMaxScaler()),
    ('kmeans',KMeans(n_clusters=5,random_state=1234))
    ])
pline.fit(X) #fit the model



##### Create & visualize the k-mean raster

In [ ]:
k_rs=pop_rs.get_bands([3,5,6]).model_predict(pline,1).set_null_value(255).astype('uint8')
k_rs2=rt.stack_bands([k_rs,baa_rs])
k_rs=k_rs2.load()
k_rs.explore(band=1)

#### Select 30 random location within each cluster and calculate mean canopy cover estimates for each iteration

In [ ]:
#convert raster cells to points

t=np.moveaxis(k_rs.to_numpy(),0,-1)
t1=t.reshape((t.shape[0]*t.shape[1],t.shape[2]))

#create a point for each cell
xs,ys=np.meshgrid(k_rs.x,k_rs.y)

#create a geodataframe
gs=gpd.GeoSeries.from_xy(xs.ravel(),ys.ravel(),crs=k_rs.crs)
tgdf=gpd.GeoDataFrame(t1,geometry=gs,columns=['cluster', 'BAA'])

#remove all null value
tgdf=tgdf[((tgdf[tgdf.columns[:-1]]<k_rs.null_value).all(axis=1))]


#split dataframe into class groups and calculate class weights based on area
c_lst=[]
for v in tgdf.cluster.unique():
    tsel=tgdf[tgdf.cluster==v]
    c_lst.append([tsel,tsel.shape[0]/tgdf.shape[0]])

#summarize canopy cover (weighted by strata area)
strata_arr=np.empty((iters,2),dtype='object')
for i in range(iters):
    s_lst=[]
    for c in c_lst:
        tsmp=c[0].sample(int(n/len(c_lst)))
        s_lst.append(tsmp.BAA.mean()*c[1])

    mv=(np.sum(s_lst))
    strata_arr[i,0]=mv
    strata_arr[i,1]='STRATA'




##### Look at the spatial distribution of one stratified random sample

In [ ]:
s_lst=[]
for c in c_lst:
    tsmp=c[0].sample(int(n/len(c_lst)))
    s_lst.append(tsmp)

tdf=(pd.concat(s_lst))
tdf.explore(column=tdf.cluster.astype('int'))

### Spread and Balanced sampling using space filling curves (Hilbert Space Filling Curves)
So what is a space filling curve? [Link to YouTube video explaining space filling curves](https://www.youtube.com/watch?v=3s7h2MHQtxc&t=1s).

To explore space filling curves and their properties let's open a Colab notebook called [Hilbert Space Filling Curves and potential ways we can use them](https://colab.research.google.com/github/jshogland/SpatialModelingTutorials/blob/main/Notebooks/hilbert_curve.ipynb)

Some key features to Hilbert space filling curves
1. Hilbert space filling curves can encode multidimensional space into one dimension and can decode one dimensional hilbert space back to the original dimensional space values.
2. Hilbert distances that are closer to one another are more similar (local proximity).
3. We can use these aspects of Hilbert space filling curves to create a well spread and balanced sample.
4. We can use Hilbert distances as a metric to find nearest neighbors.

#### Creating a well spread and balanced sample
- Use hilbert space filling curves to encode our ancillary data into one dimension represented by hilbert distance
- Sort hilbert distances to order our data in one dimension space
- Use random systematic sampling to select our locations from our sorted hilbert distances.

##### Normalize the ancillary data raster to a value between 0 and 1

In [ ]:
#normalize ancillary data rasters to a value between 0 and 1
p=8 # precision of the hilbert curve
pred_rs=pop_rs.get_bands([3,5,6]) #only using the bands that were related to BAA
rs_lst=[]
for b in range(pred_rs.nbands):
    brs=pred_rs.get_bands([b+1])
    vmin=brs.min().compute()
    vmax=brs.max().compute()
    nbrs=(((brs-vmin)/(vmax-vmin)*2**p)-1).set_null_value(255).astype('uint8')
    rs_lst.append(nbrs)

rs_lst.append(pop_rs.get_bands([13]).astype('uint8'))

npred_rs=rt.stack_bands(rs_lst).load()


##### Convert raster to points and remove null values

In [ ]:
#format data to dataframe array
t=np.moveaxis(npred_rs.to_numpy(),0,-1)
t1=t.reshape((t.shape[0]*t.shape[1],t.shape[2]))

#create a point for each cell
xs,ys=np.meshgrid(npred_rs.x,npred_rs.y)

#create a geodataframe
gs=gpd.GeoSeries.from_xy(xs.ravel(),ys.ravel(),crs=npred_rs.crs)
tgdf=gpd.GeoDataFrame(t1,geometry=gs,columns=['B04', 'B11','B12','baa'])

#remove all null value
tgdf=tgdf[((tgdf[tgdf.columns[:-1]]<npred_rs.null_value).all(axis=1))]

##### Create the Hilbert model, apply it to the normalized ancillary data, and sort based on distance

In [ ]:
#subset data to normalized values only
X=tgdf[tgdf.columns[:-2]]

#create Hilbert model and specify how many cpu's to use when
#calulating distance
hc=HilbertCurve(p=p,n=X.shape[1])#for faster processing set to as manny processors as your system can handle want, e.g.,  n_procs=6)

#calculate the distance for each record
vls=hc.distances_from_points(X.values)

#add distance to geodataframe and sort values
tgdf['dist']=vls
s_gdf=tgdf.sort_values('dist').reset_index(drop=True)


##### Look at the spatial distribution of one space filling curve selection

In [ ]:
#select records using systematic random sampling
N2=s_gdf.shape[0] #we have some cells that were dropped along the edge to calculated slope, northing, easting, and curvature
sk=N2/n #in a sense we are stratifying by groups of 150. Because distances are ordered and closer distances are more similar to one another, we are grouping by similar values
rstart=(np.random.random(1)*sk).astype('uint64')[0] #create a random start for sampling
r_index=np.arange(rstart,N2,sk).astype('uint64') #create index of sample observations
s_gdf.loc[r_index].explore()

##### Select 1000 well spread and balanced samples using hilbert distance and summarize mean Canopy Cover

In [ ]:
h_arr=np.empty((iters,2),dtype='object')
for i in range(iters):
    rstart=(np.random.random(1)*sk).astype('uint64')[0] #create a random start for sampling
    r_index=np.arange(rstart,N2,sk).astype('uint64') #create index of sample observations
    h_arr[i,0]=s_gdf.iloc[r_index].baa.mean()
    h_arr[i,1]='HILBERT'

##### Display box plot and histogram of HILBERT, SRS, SYS, and STRATA means

In [ ]:
df_vls=np.concat([srs_arr,sys_arr,strata_arr,h_arr])
df=pd.DataFrame(df_vls,columns=['mean_baa','stype'])
df['stype']=df.stype.astype('string')
df['mean_baa']=df.mean_baa.astype('float')
df_grp=df.groupby('stype')
print('Population mean BAA:',mean_baa)
display(df_grp.mean_baa.agg(['mean','std','count']))
p1=df.plot(kind='box',by='stype',figsize=(15,5))
p1.iloc[0].scatter(1, mean_baa, color='red', marker='o', s=100, label=f"Mean: {mean_baa:.2f}")
p1.iloc[0].scatter(2, mean_baa, color='red', marker='o', s=100, label=f"Mean: {mean_baa:.2f}")
p1.iloc[0].scatter(3, mean_baa, color='red', marker='o', s=100, label=f"Mean: {mean_baa:.2f}")
p1.iloc[0].scatter(4, mean_baa, color='red', marker='o', s=100, label=f"Mean: {mean_baa:.2f}")
p1.iloc[0].set_title('HILBERT - SRS - STRATA - SYS')
display(p1)


In [ ]:
p2=df.groupby('stype')['mean_baa'].plot(kind='hist', legend=True,alpha=0.4,figsize=(15,5))
display(p2)

### Perform a 2 sample KS test for both distance and BAA and plot the distribution of BAA for the sample distributions.
#### Get one sample for each approach

In [ ]:
#Get samples
tdf=pd.DataFrame(vls_baa,columns=['BAA'])
tdf['stype']='POP'

pop_smp=tdf.sample(10000) #large sample for graphing

#SRS
srs_smp=tdf.sample(150)
srs_smp['stype']='SRS'

#SYS
tarea=bndp.area.values
carea=tarea/n
sk=carea**0.5
xmin,ymin,xmax,ymax=bndp.total_bounds
rsx,rsy=np.random.random(2)*sk[0]
xs=np.arange(xmin+rsx,xmax,sk[0])
ys=np.arange(ymin+rsy,ymax,sk[0])
xs2,ys2=np.meshgrid(xs,ys)
geom=gpd.GeoSeries.from_xy(xs2.ravel(),ys2.ravel(),crs=bndp.crs)
geoc=geom.clip(bndp)
sys_smp=zonal.extract_points_eager(geoc,baa_rs,'BAA',axis=1).compute()[['BAA_1']]
sys_smp.columns=['BAA']
sys_smp['stype']='SYS'

#STRATA
s_lst=[]
for c in c_lst:
    tsmp=c[0].sample(int(n/len(c_lst)))
    s_lst.append(tsmp)
strata_smp=pd.concat(s_lst)[['BAA']]
strata_smp['stype']='STRATA'


#Hilbert
hilb_smp=s_gdf.iloc[r_index][['baa']] #last sample
hilb_smp['stype']='HILBERT'
hilb_smp.columns=['BAA','stype']

#combine all samples
adf=pd.concat([pop_smp,srs_smp,sys_smp,strata_smp,hilb_smp])

##### KS statistics and display distributions

In [ ]:
#look at KS statistic for hilbert distance values
dks_test=scipy.stats.ks_2samp(vls_baa,hilb_smp.BAA) #KS 2 sample test
print('KS test stat for BAA =',dks_test.statistic,'\nKS pvalue for BAA =',dks_test.pvalue)

#plot distributions of samples
p2=adf.groupby('stype')['BAA'].plot(kind='kde', legend=True,linewidth=1,figsize=(15,5),title='Sample Spreads')
display(p2)

### Hilbert space filing curves & sampling
Hilbert space filling curves can be used to create well spread and balanced samples that both captures the population mean and that mimic the naturally occurring distribution of the response. When using Hilbert Curves to crate a well spread and balanced sample, strong relationships between response and ancillary values will produce samples that have distribution more similar to the population, with less variability among samples. If the relationship between response and predictor variables is week, the sample distribution reverts to the same as a simple random sample.

#### How can we use Hilbert Distance to address accessibility issues?